# 10 — Silver Contracts & Processes

**Objetivo:** construir la primera representación canónica Silver de contratos y procesos, preservando lineage, segregando estado mutable y eliminando la explosión de joins observada en Bronze.

Este notebook **no crea features ML ni labels**. La lógica reusable vive en `src/argos/silver/`; aquí se orquesta, inspecciona evidencia y se valida el resultado.


## Invariantes heredados de M1

- `contracts.id_contrato` es único y no nulo.
- `contracts.proceso_de_compra -> processes.id_del_portafolio` es la relación empíricamente aceptada.
- `processes.id_del_portafolio` no es único en Bronze: existen múltiples payloads/observaciones por identidad.
- Los campos contractuales mutables/post-`t0` deben quedar físicamente separados.
- Silver **no equivale** a una whitelist de features; Gold volverá a exigir point-in-time correctness.


In [ ]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files
from argos.config.silver import load_silver_spec
from argos.evaluation.m1_gate import evaluate_m1_gate
from argos.silver.contracts_processes import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

gate_path = paths.docs / "decisions" / "M1_GATE.md"
if gate_path.exists():
    gate_closed = "M1 ✅ CLOSED" in gate_path.read_text(encoding="utf-8")
else:
    _, gate_decision = evaluate_m1_gate(paths.reports, PROJECT_CONFIG)
    gate_closed = gate_decision.closed
assert gate_closed, "M1 no está formalmente cerrado; no iniciar Silver."

DATASETS, _ = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)

{
    "contracts": FILES["contracts"].name,
    "processes": FILES["processes"].name,
    "silver_schema_version": SILVER_SPEC.schema_version,
    "build_implementation": BUILD_IMPLEMENTATION,
    "duckdb_threads": SILVER_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": SILVER_SPEC.runtime.duckdb_memory_limit,
    "m1_gate_source": str(gate_path) if gate_path.exists() else "re-evaluated from M1 reports",
}


## 1. Plan de routing de columnas

Antes de escribir Parquet inspeccionamos qué columnas irán a `contracts.parquet`, cuáles se segregan en `contracts_snapshot_state.parquet` y cuáles se excluyen del Silver analítico general por sensibilidad.


In [ ]:
from argos.silver.contracts_processes import plan_contracts_processes_silver

routing = plan_contracts_processes_silver(
    FILES["contracts"],
    FILES["processes"],
    column_profile_path=paths.reports / "profiling" / "column_profile.parquet",
    silver_spec=SILVER_SPEC,
    sample_size=PROJECT_CONFIG["profiling"]["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

routing.groupby(["dataset", "route"]).size().rename("columns").reset_index()


In [ ]:
routing.loc[
    (routing["dataset"] == "contracts")
    & (routing["route"].isin(["snapshot_state", "excluded_sensitive"])),
    [
        "raw_column",
        "canonical_column",
        "semantic_type",
        "leakage_risk_hint",
        "route",
        "reason",
    ],
].sort_values(["route", "canonical_column"])


## 2. Construcción Silver

La construcción produce cinco artefactos:

- `contracts.parquet`: contrato canónico, sin los campos de snapshot mutable conocidos.
- `contracts_snapshot_state.parquet`: estado actual/post-`t0` segregado.
- `process_versions.parquet`: payloads distintos de procesos después de deduplicar filas Bronze exactamente iguales.
- `processes.parquet`: identidad canónica `1 fila = 1 process_id` + diagnósticos de variantes.
- `contract_process_links.parquet`: `1 fila = 1 contrato`, sin expansión del join.


### Ejecución out-of-core

`processes` se deduplica mediante staging Parquet + índice estrecho de fingerprints; no se ejecuta un `GROUP BY ALL` ancho sobre el CSV completo. La configuración de Silver limita DuckDB a 2 threads y 8 GB para dejar headroom al kernel de Jupyter y permitir spill a `.cache/duckdb`.


**Implementación esperada:** `m2-10-out-of-core-v2`. Si la primera celda muestra otra implementación o no muestra `duckdb_memory_limit = 8GB`, el kernel está cargando una versión anterior del paquete.


In [ ]:
from argos.silver.contracts_processes import build_contracts_processes_silver

result = build_contracts_processes_silver(
    FILES["contracts"],
    FILES["processes"],
    contracts_spec=DATASETS["contracts"],
    processes_spec=DATASETS["processes"],
    column_profile_path=paths.reports / "profiling" / "column_profile.parquet",
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    silver_spec=SILVER_SPEC,
    date_order=PROJECT_CONFIG["temporal"]["date_order"],
    sample_size=PROJECT_CONFIG["profiling"]["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}


## 3. Auditoría de conversión de tipos

Para fechas y dinero Silver conserva tres representaciones: valor tipado, `__raw` y `__parse_status`. Los valores no reconocidos no desaparecen silenciosamente.


In [ ]:
result.conversion_audit.sort_values(
    ["unparsed_rows", "dataset", "canonical_column"],
    ascending=[False, True, True],
)


**Interpretación:** un `unparsed_rows > 0` no se oculta ni se imputa. Si corresponde a un campo de snapshot mutable/deferred, queda registrado con su valor `__raw` para resolución posterior. Para `fecha_de_firma` sí exigimos que la población parseada reproduzca la evidencia temporal de M1, porque define `t0`.


## 4. Validación del componente 10

Estos checks son **bloqueantes para continuar a `11_silver_suppliers_bidders.ipynb`**. No son todavía el gate completo de M2; ese rol pertenece a `15_m2_gate.ipynb`.


In [ ]:
result.validation


In [ ]:
from argos.common.duckdb import connect
from argos.common.sql import sql_string

con = connect(
    temp_directory=paths.root / ".cache" / "duckdb",
    threads=SILVER_SPEC.runtime.duckdb_threads,
    memory_limit=SILVER_SPEC.runtime.duckdb_memory_limit,
)
try:
    link_status = con.execute(
        f"""
        SELECT link_status, COUNT(*) AS contracts
        FROM read_parquet({sql_string(result.artifacts['contract_process_links'])})
        GROUP BY link_status
        ORDER BY contracts DESC
        """
    ).fetchdf()

    process_variants = con.execute(
        f"""
        SELECT
            COUNT(*) AS processes,
            AVG(version_count) AS avg_versions,
            MAX(version_count) AS max_versions,
            COUNT(*) FILTER (WHERE version_count > 1) AS processes_with_multiple_versions
        FROM read_parquet({sql_string(result.artifacts['processes'])})
        """
    ).fetchdf()
finally:
    con.close()

display(link_status)
display(process_variants)


In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()

print("10 ✅ PASS" if result.ready_for_next_notebook else "10 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))

assert result.ready_for_next_notebook, (
    "Notebook 10 remains OPEN. Resolve the failed checks before notebook 11."
)


## Criterio de salida

`10` termina únicamente si los checks bloqueantes pasan y existen los cinco Parquet Silver más el manifest.

**Siguiente paso:** `11_silver_suppliers_bidders.ipynb`. La relación directa bidder→supplier permanece sin resolver y se cerrará explícitamente en `13_entity_resolution.ipynb`; no se fuerza un join artificial en este notebook.
